In [13]:
# ══════════════════════════════════════════════════════════════════
# NOTEBOOK 2: PROPHET FORECASTING
# Trains Prophet models on baseline period Jan2021-Dec2025
# Validates on Jan-Jun 2026 where data is complete
# Forecasts Jul-Sep 2026
#
# Special handling:
#   Demand: excluded from Prophet
#           Use 3-month rolling avg of last complete months
#           completeness threshold: >= 90%
#
# Models trained (12 total, not 13):
#   STU deviation      → 3 models (Wheat, Corn, Rice)
#   BDI ratio          → 1 model  (shared)
#   PPI deviation      → 3 models (Wheat, Corn, Rice)
#   KSA deviation      → 3 models (Wheat, Corn, Rice)
#   Demand             → rule-based rolling avg (no Prophet)
#   Policy             → rule-based decay (no Prophet)
#
# Output tables:
#   srm.prophet_forecasts
#   srm.prophet_validation
#   srm.prophet_model_metrics
#   srm.prophet_current_values
#   srm.prophet_demand_current  ← new: demand rolling avg
# ══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

try:
    from prophet import Prophet
    print("Prophet already installed ✓")
except ImportError:
    import subprocess
    subprocess.run(["pip","install","prophet","--quiet","--break-system-packages"], check=True)
    from prophet import Prophet
    print("Prophet installed ✓")

COMMODITIES    = ["Wheat","Corn","Rice","Soybean","Barley"]
BASELINE_START = pd.Timestamp("2021-01-01")
BASELINE_END   = pd.Timestamp("2025-12-01")
SCORING_END    = pd.Timestamp("2026-07-01")
FORECAST_MONTHS = pd.to_datetime(
    ["2026-08-01","2026-09-01","2026-10-01"]
).astype("datetime64[us]")

DEMAND_COMPLETENESS_THRESHOLD = 0.90  # 90% reporting completeness

def save_to_lakehouse(df_pandas, table_name, schema="srm"):
    full_name = f"{schema}.{table_name}"
    spark.createDataFrame(df_pandas) \
         .write.mode("overwrite") \
         .option("overwriteSchema","true") \
         .format("delta") \
         .saveAsTable(full_name)
    count = spark.table(full_name).count()
    print(f"✓ {full_name}: {count} rows saved")

def load_table(table_name, drop_strings=False):
    STRING_COLS = [
        "ksa_top1_country","ksa_top1_country_lag1",
        "ksa_top3_countries","top_buyer_country",
        "individually_tracked_countries"
    ]
    df_spark = spark.table(f"srm.{table_name}")
    if drop_strings:
        drop_cols = [c for c in STRING_COLS if c in df_spark.columns]
        if drop_cols:
            df_spark = df_spark.drop(*drop_cols)
    df = df_spark.toPandas()
    for col in df.columns:
        if col in ["ds","year_month"]:
            df[col] = pd.to_datetime(df[col])
    return df

print("=== Notebook 2: Prophet Models ===")
print(f"Training:   {BASELINE_START.date()} → {BASELINE_END.date()}")
print(f"Validation: Jan 2026 → {SCORING_END.date()}")
print(f"Forecast:   {FORECAST_MONTHS[0].strftime('%b %Y')} → {FORECAST_MONTHS[-1].strftime('%b %Y')}")
print(f"Models:     25 Prophet + 1 demand rolling avg + 1 policy rule-based")   # was 20

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 17, Finished, Available, Finished, False)

Prophet already installed ✓
=== Notebook 2: Prophet Models ===
Training:   2021-01-01 → 2025-12-01
Validation: Jan 2026 → 2026-07-01
Forecast:   Aug 2026 → Oct 2026
Models:     25 Prophet + 1 demand rolling avg + 1 policy rule-based


In [14]:
# ══════════════════════════════════════════════════════════════════
# STEP 1: LOAD TIME SERIES TABLES
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 1: Loading time series tables ===")

df_stu_ts    = load_table("prophet_ts_stu")
df_bdi_ts    = load_table("prophet_ts_bdi")
df_ppi_ts    = load_table("prophet_ts_ppi")
df_ksa_ts    = load_table("prophet_ts_ksa")
df_policy_ts = load_table("prophet_ts_policy")
df_price_ts  = load_table("prophet_ts_price")   # NEW
df_price_rolling_ts = load_table("prophet_ts_price_rolling")   # NEW — load the rolling-baseline signal for this diagnostic


# Load raw demand table for rolling avg calculation
df_demand_raw = load_table("features_demand", drop_strings=True)
df_demand_raw["year_month"] = pd.to_datetime(df_demand_raw["year_month"])
df_baseline   = load_table("prophet_baseline_summary")

print(f"STU:    {df_stu_ts.shape}")
print(f"BDI:    {df_bdi_ts.shape}")
print(f"PPI:    {df_ppi_ts.shape}")
print(f"KSA:    {df_ksa_ts.shape}")
print(f"Policy: {df_policy_ts.shape}")
print(f"Demand raw: {df_demand_raw.shape}")
print(f"Price:  {df_price_ts.shape}")   # NEW
print(f"Price (rolling): {df_price_rolling_ts.shape}")

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 18, Finished, Available, Finished, False)


=== Step 1: Loading time series tables ===
STU:    (347, 7)
BDI:    (450, 10)
PPI:    (287, 7)
KSA:    (360, 7)
Policy: (360, 7)
Demand raw: (289, 56)
Price:  (524, 7)
Price (rolling): (469, 9)


In [15]:
# ══════════════════════════════════════════════════════════════════
# DIAGNOSTIC: FOB Price — rolling-baseline signal, changepoint_prior_scale grid search
# Compares against log-price and original fixed-baseline results already on record.
# ══════════════════════════════════════════════════════════════════

CANDIDATE_SCALES = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]

print("\n=== FOB Price (rolling baseline) changepoint_prior_scale grid search ===")
print(f"{'Commodity':<10}{'Scale':>8}{'RMSE ($)':>12}{'MAE ($)':>12}{'Jul Residual':>15}")
print("-" * 60)

rolling_tuning_results = []

for commodity in COMMODITIES:
    for scale in CANDIDATE_SCALES:
        val, fore, metrics = train_prophet(
            df_price_rolling_ts, commodity, "fob_price_deviation_rolling",
            yearly_seasonality=True,
            changepoint_prior_scale=scale,
            seasonality_prior_scale=10.0
        )
        if val is None or val.empty:
            continue

        val_sorted = val.sort_values("ds")
        # y here is (price - rolling_baseline_at_that_month), so reconstructing
        # actual/predicted price requires adding back each row's OWN rolling baseline,
        # not a single constant — this is the key structural difference vs. log-price.
        rb = df_price_rolling_ts[
            (df_price_rolling_ts["commodity"]==commodity)
        ].set_index("ds")["rolling_baseline"]

        actual_price    = val_sorted["y"].values + rb.reindex(val_sorted["ds"]).values
        predicted_price = val_sorted["yhat"].values + rb.reindex(val_sorted["ds"]).values
        residuals = actual_price - predicted_price

        rmse = np.sqrt(np.mean(residuals**2))
        mae  = np.mean(np.abs(residuals))
        jul_residual = residuals[-1]

        rolling_tuning_results.append({
            "commodity": commodity, "scale": scale,
            "rmse": rmse, "mae": mae, "jul_residual": jul_residual
        })
        print(f"{commodity:<10}{scale:>8.2f}{rmse:>12.2f}{mae:>12.2f}{jul_residual:>15.2f}")

    print("-" * 60)

df_rolling_tuning = pd.DataFrame(rolling_tuning_results)
print("\n=== Best scale per commodity (lowest RMSE) — rolling baseline ===")
best_rolling = df_rolling_tuning.loc[df_rolling_tuning.groupby("commodity")["rmse"].idxmin()]
print(best_rolling[["commodity","scale","rmse","mae","jul_residual"]].to_string(index=False))

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 19, Finished, Available, Finished, False)

02:16:06 - cmdstanpy - INFO - Chain [1] start processing
02:16:06 - cmdstanpy - INFO - Chain [1] start processing
02:16:08 - cmdstanpy - INFO - Chain [1] done processing
02:16:08 - cmdstanpy - INFO - Chain [1] start processing
02:16:09 - cmdstanpy - INFO - Chain [1] done processing
02:16:09 - cmdstanpy - INFO - Chain [1] start processing
02:16:10 - cmdstanpy - INFO - Chain [1] done processing
02:16:10 - cmdstanpy - INFO - Chain [1] start processing
02:16:12 - cmdstanpy - INFO - Chain [1] done processing
02:16:12 - cmdstanpy - INFO - Chain [1] start processing
02:16:13 - cmdstanpy - INFO - Chain [1] done processing
02:16:13 - cmdstanpy - INFO - Chain [1] start processing
02:16:15 - cmdstanpy - INFO - Chain [1] done processing
02:16:15 - cmdstanpy - INFO - Chain [1] start processing


    RMSE=17.115  MAE=15.761  MAPE=1351.2%
    Forecast: [-16.199 -16.631 -17.051]
Wheat         0.15       17.11       15.76           9.06

  fob_price_deviation_rolling | Wheat
    Train: 89 rows (2018-08-01 → 2025-12-01)
    Val:   7 rows
    RMSE=40.148  MAE=32.775  MAPE=261.3%
    Forecast: [38.504 40.453 42.341]
Barley        0.50       40.15       32.77         -66.14
------------------------------------------------------------

=== Best scale per commodity (lowest RMSE) — rolling baseline ===
commodity  scale      rmse       mae  jul_residual
   Barley   0.15 19.948188 17.484255    -30.595394
     Corn   0.30  5.109429  4.432347     -0.730997
     Rice   0.50 75.143043 67.238570    121.827050
  Soybean   0.50 14.740241 11.729147     27.942901
    Wheat   0.50 13.134973 10.809219    -22.399741


In [3]:
# ══════════════════════════════════════════════════════════════════
# STEP 2: DEFINE PROPHET TRAINING FUNCTION
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 2: Defining Prophet training function ===")

def train_prophet(
    df_ts,
    commodity,
    indicator_name,
    yearly_seasonality=True,
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10.0
):
    """
    Train Prophet on baseline period (Jan2021-Dec2025).
    Validate on Jan-Jun 2026.
    Forecast Jul-Sep 2026.
    """
    # Filter commodity
    if commodity != "ALL":
        dc = df_ts[df_ts["commodity"] == commodity].copy()
    else:
        dc = df_ts.drop_duplicates(subset=["ds"]).copy()

    dc = dc.sort_values("ds").reset_index(drop=True)
    dc["ds"] = dc["ds"].astype("datetime64[us]")

    # Training data: baseline period only
    df_train = dc[dc["ds"] <= BASELINE_END.as_unit("us")][["ds","y"]].copy()

    # Validation data: Jan-Jun 2026
    val_start = pd.Timestamp("2026-01-01").as_unit("us")
    val_end   = SCORING_END.as_unit("us")
    df_val    = dc[(dc["ds"] >= val_start) & (dc["ds"] <= val_end)][["ds","y"]].copy()

    print(f"\n  {indicator_name} | {commodity}")
    print(f"    Train: {len(df_train)} rows ({df_train['ds'].min().date()} → {df_train['ds'].max().date()})")
    print(f"    Val:   {len(df_val)} rows")

    if len(df_train) < 12:
        print(f"    SKIP: insufficient training data")
        return None, None, None

    # Train Prophet
    model = Prophet(
        yearly_seasonality=yearly_seasonality,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=changepoint_prior_scale,
        seasonality_prior_scale=seasonality_prior_scale,
        interval_width=0.80,
        uncertainty_samples=500
    )
    model.fit(df_train)

    # Build future dates — val + forecast
    if len(df_val) > 0:
        val_dates = df_val[["ds"]].copy()
        val_dates["ds"] = val_dates["ds"].astype("datetime64[us]")
    else:
        val_dates = pd.DataFrame({"ds": pd.Series([], dtype="datetime64[us]")})

    fore_dates = pd.DataFrame({"ds": FORECAST_MONTHS})
    all_future = pd.concat([val_dates, fore_dates], ignore_index=True) \
                   .drop_duplicates().reset_index(drop=True)

    future_df = model.make_future_dataframe(periods=0, freq="MS")
    future_df["ds"] = future_df["ds"].astype("datetime64[us]")
    future_df = pd.concat([future_df, all_future], ignore_index=True) \
                  .drop_duplicates(subset=["ds"]).sort_values("ds").reset_index(drop=True)

    forecast = model.predict(future_df)
    forecast["ds"] = forecast["ds"].astype("datetime64[us]")

    # Validation metrics
    metrics = {
        "indicator": indicator_name, "commodity": commodity,
        "train_rows": len(df_train), "val_rows": len(df_val),
        "RMSE": np.nan, "MAE": np.nan, "MAPE": np.nan
    }

    df_validation = None
    if len(df_val) > 0:
        vm = df_val.merge(
            forecast[["ds","yhat","yhat_lower","yhat_upper"]], on="ds", how="left"
        )
        vm["residual"]  = vm["y"] - vm["yhat"]
        vm["abs_error"] = vm["residual"].abs()
        vm["pct_error"] = vm["abs_error"] / (vm["y"].abs() + 1e-6) * 100

        metrics["RMSE"] = round(np.sqrt((vm["residual"]**2).mean()), 4)
        metrics["MAE"]  = round(vm["abs_error"].mean(), 4)
        metrics["MAPE"] = round(vm["pct_error"].mean(), 2)

        df_validation = vm.copy()
        df_validation["indicator"]  = indicator_name
        df_validation["commodity"]  = commodity

        print(f"    RMSE={metrics['RMSE']:.3f}  MAE={metrics['MAE']:.3f}  MAPE={metrics['MAPE']:.1f}%")
    else:
        print(f"    No validation data")

    # Forecast Jul-Sep 2026
    df_forecast = forecast[forecast["ds"].isin(FORECAST_MONTHS)][[
        "ds","yhat","yhat_lower","yhat_upper"
    ]].copy()
    df_forecast["indicator"] = indicator_name
    df_forecast["commodity"] = commodity
    df_forecast["month_label"] = df_forecast["ds"].dt.strftime("%b %Y")

    print(f"    Forecast: {df_forecast['yhat'].values.round(3)}")

    return df_validation, df_forecast, metrics

print("✓ Prophet training function defined")

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 5, Finished, Available, Finished, False)


=== Step 2: Defining Prophet training function ===
✓ Prophet training function defined


In [4]:
# ══════════════════════════════════════════════════════════════════
# STEP 3: TRAIN ALL PROPHET MODELS
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 3: Training Prophet models ===")

all_validations = []
all_forecasts   = []
all_metrics     = []

# ── 3a: STU deviation — 3 models ──────────────────────────────────
print("\n--- STU Deviation (5 models) ---")
for commodity in COMMODITIES:
    val, fore, metrics = train_prophet(
        df_stu_ts, commodity, "stu_deviation",
        yearly_seasonality=True,
        changepoint_prior_scale=0.05  # slow moving
    )
    if val is not None:     all_validations.append(val)
    if fore is not None:    all_forecasts.append(fore)
    if metrics is not None: all_metrics.append(metrics)

# ── 3b: BDI ratio — 1 model replicated ────────────────────────────
print("\n--- BDI Ratio (1 model, replicated) ---")
val, fore, metrics = train_prophet(
    df_bdi_ts, "Wheat", "bdi_ratio",
    yearly_seasonality=True,
    changepoint_prior_scale=0.20,  # volatile
    seasonality_prior_scale=15.0
)
# NEW — replace ["Corn","Rice"] with a derived list, all 3 places
bdi_replicate_to = [c for c in COMMODITIES if c != "Wheat"]  # was hardcoded ["Corn","Rice"]

if fore is not None:
    for commodity in bdi_replicate_to:
        fore_copy = fore.copy()
        fore_copy["commodity"] = commodity
        all_forecasts.append(fore_copy)
    all_forecasts.append(fore)

if val is not None:
    for commodity in bdi_replicate_to:
        val_copy = val.copy()
        val_copy["commodity"] = commodity
        all_validations.append(val_copy)
    all_validations.append(val)

if metrics is not None:
    for commodity in bdi_replicate_to:
        m_copy = metrics.copy()
        m_copy["commodity"] = commodity
        all_metrics.append(m_copy)
    all_metrics.append(metrics)

# ── 3c: PPI deviation — 5 models ─────────────────────────────────
print("\n--- PPI Deviation (5 models) ---")
for commodity in COMMODITIES:
    val, fore, metrics = train_prophet(
        df_ppi_ts, commodity, "ppi_deviation",
        yearly_seasonality=True,
        changepoint_prior_scale=0.10
    )
    if val is not None:     all_validations.append(val)
    if fore is not None:    all_forecasts.append(fore)
    if metrics is not None: all_metrics.append(metrics)

# ── 3d: KSA deviation — 5 models ─────────────────────────────────
print("\n--- KSA Deviation (5 models) ---")
for commodity in COMMODITIES:
    val, fore, metrics = train_prophet(
        df_ksa_ts, commodity, "ksa_deviation",
        yearly_seasonality=False,      # annual data
        changepoint_prior_scale=0.01,  # very slow
        seasonality_prior_scale=1.0
    )
    if val is not None:     all_validations.append(val)
    if fore is not None:    all_forecasts.append(fore)
    if metrics is not None: all_metrics.append(metrics)

# # ── 3e: FOB Price deviation — 5 models ────────────────────────────
# print("\n--- FOB Price Deviation (5 models) ---")
# for commodity in COMMODITIES:
#     val, fore, metrics = train_prophet(
#         df_price_ts, commodity, "fob_price_deviation",
#         yearly_seasonality=True,
#         changepoint_prior_scale=0.05,
#         seasonality_prior_scale=10.0
#     )
#     if val is not None:     all_validations.append(val)
#     if fore is not None:    all_forecasts.append(fore)
#     if metrics is not None: all_metrics.append(metrics)

# ── 3e: FOB Price deviation — 5 models (per-commodity tuned) ──────
# Unlike STU/PPI/KSA, price series show genuinely different trend
# behavior per commodity — a single changepoint_prior_scale badly
# under/over-fits some of them. Values below are starting points from
# comparing 0.05/0.15/0.30 against the actual Jan-Jul 2026 validation
# window; revisit if the full RMSE/MAPE table still looks weak.
# PRICE_CHANGEPOINT_SCALE = {
#     "Wheat":   0.30,
#     "Corn":    0.15,
#     "Rice":    0.10,   # tied between 0.05 and 0.15 on July residual — splitting the difference
#     "Soybean": 0.05,
#     "Barley":  0.15,
# }

PRICE_CHANGEPOINT_SCALE = {
    "Wheat":   0.05,   # RESET — old tuned value (0.30) was fit to the raw-deviation signal, not log-price
    "Corn":    0.05,   # RESET — was 0.15
    "Rice":    0.05,   # RESET — was 0.10
    "Soybean": 0.05,   # RESET — was 0.05 (unchanged, but re-verify)
    "Barley":  0.05,   # RESET — was 0.15
}


print("\n--- FOB Price Deviation (5 models, per-commodity tuned) ---")
for commodity in COMMODITIES:
    val, fore, metrics = train_prophet(
        df_price_ts, commodity, "fob_price_deviation",
        yearly_seasonality=True,
        changepoint_prior_scale=PRICE_CHANGEPOINT_SCALE[commodity],
        seasonality_prior_scale=10.0
    )
    if val is not None:     all_validations.append(val)
    if fore is not None:    all_forecasts.append(fore)
    if metrics is not None: all_metrics.append(metrics)

print(f"\nTotal forecast records:   {sum(len(f) for f in all_forecasts)}")
print(f"Total validation records: {sum(len(v) for v in all_validations)}")
print(f"Total metrics records:    {len(all_metrics)}")

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 6, Finished, Available, Finished, False)

01:37:36 - cmdstanpy - INFO - Chain [1] start processing
01:37:37 - cmdstanpy - INFO - Chain [1] done processing
01:37:37 - cmdstanpy - INFO - Chain [1] done processing
01:37:39 - cmdstanpy - INFO - Chain [1] done processing
01:37:39 - cmdstanpy - INFO - Chain [1] start processing


    RMSE=0.438  MAE=0.397  MAPE=23.6%
    Forecast: [1.825 1.94  2.054]

--- BDI Ratio (1 model, replicated) ---

  bdi_ratio | Wheat
    Train: 82 rows (2019-03-01 → 2025-12-01)
    Val:   7 rows
    No validation data
    Forecast: [-5.726 -5.775 -5.823]

--- FOB Price Deviation (5 models, per-commodity tuned) ---

  fob_price_deviation | Wheat
    Train: 100 rows (2017-09-01 → 2025-12-01)
    Val:   7 rows
    RMSE=0.027  MAE=0.024  MAPE=0.5%
    Forecast: [5.467 5.468 5.469]

  fob_price_deviation | Corn
    Train: 100 rows (2017-09-01 → 2025-12-01)
    Val:   7 rows
    RMSE=0.024  MAE=0.021  MAPE=0.4%
    Forecast: [5.348 5.35  5.352]

  fob_price_deviation | Rice
    Train: 84 rows (2019-01-01 → 2025-12-01)
    Val:   7 rows
    RMSE=0.361  MAE=0.314  MAPE=5.6%
    Forecast: [5.071 5.022 4.973]

  fob_price_deviation | Soybean
    Train: 100 rows (2017-09-01 → 2025-12-01)
    Val:   7 rows
    RMSE=0.111  MAE=0.096  MAPE=1.6%
    Forecast: [5.989 5.985 5.982]

  fob_price_deviat

In [5]:
# ══════════════════════════════════════════════════════════════════
# STEP 4: DEMAND — ROLLING AVG OF LAST COMPLETE MONTHS
# No Prophet — use 3-month rolling avg of complete months
# Complete = demand_data_completeness >= 90%
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 4: Demand — 3-month rolling avg (no Prophet) ===")

demand_rows = []

for commodity in COMMODITIES:
    dc = df_demand_raw[df_demand_raw["commodity"] == commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)

    # Filter to complete months only
    dc_complete = dc[dc["demand_data_completeness"] >= DEMAND_COMPLETENESS_THRESHOLD].copy()

    print(f"\n{commodity}:")
    print(f"  Total months available: {len(dc)}")
    print(f"  Complete months (>={DEMAND_COMPLETENESS_THRESHOLD*100:.0f}%): {len(dc_complete)}")

    if dc_complete.empty:
        print(f"  WARNING: No complete months found")
        continue

    # Get baseline avg from summary table
    baseline_row = df_baseline[
        (df_baseline["commodity"]==commodity) &
        (df_baseline["indicator"]=="demand_ratio")
    ]
    baseline_avg = float(baseline_row["baseline_avg"].values[0]) if not baseline_row.empty else None

    # Last complete month
    last_complete = dc_complete.iloc[-1]
    print(f"  Last complete month: {last_complete['year_month'].date()} "
          f"(completeness: {last_complete['demand_data_completeness']:.1%})")
    print(f"  Volume: {last_complete['driver_total_mt']/1e6:.2f}M MT")

    # 3-month rolling avg of last 3 complete months
    last3_complete = dc_complete.tail(3)
    rolling_avg_mt = last3_complete["driver_total_mt"].mean()
    rolling_ratio  = rolling_avg_mt / baseline_avg if baseline_avg else np.nan

    print(f"  3-month rolling avg (complete months): {rolling_avg_mt/1e6:.2f}M MT")
    print(f"  Ratio vs 5Y baseline:                  {rolling_ratio:.3f}x")

    # Use this ratio for current AND forecast months
    # Assumption: demand stays at recent complete level
    # Jul-Sep 2026 = same ratio as last 3 complete months
    all_periods = pd.to_datetime([SCORING_END] + list(FORECAST_MONTHS)).astype("datetime64[us]")

    for period in all_periods:
        is_forecast = period > SCORING_END.as_unit("us")
        demand_rows.append({
            "ds":           period,
            "yhat":         round(rolling_ratio, 4),
            "yhat_lower":   round(rolling_ratio * 0.85, 4),  # ±15% uncertainty
            "yhat_upper":   round(rolling_ratio * 1.15, 4),
            "y_raw_mt":     round(rolling_avg_mt, 2),
            "baseline_avg": round(baseline_avg, 2) if baseline_avg else np.nan,
            "indicator":    "demand_ratio",
            "commodity":    commodity,
            "method":       "rolling_avg_3m_complete",
            "period_type":  "forecast" if is_forecast else "current"
        })

df_demand_current = pd.DataFrame(demand_rows)
df_demand_current["ds"] = pd.to_datetime(df_demand_current["ds"])

print(f"\nDemand rolling avg table: {df_demand_current.shape}")
print(f"\nDemand ratio summary (rolling avg of complete months):")
print(df_demand_current[df_demand_current["period_type"]=="current"][[
    "commodity","yhat","y_raw_mt","baseline_avg"
]].to_string(index=False))

save_to_lakehouse(df_demand_current, "prophet_demand_current")

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 7, Finished, Available, Finished, False)


=== Step 4: Demand — 3-month rolling avg (no Prophet) ===

Wheat:
  Total months available: 53
  Complete months (>=90%): 36
  Last complete month: 2024-12-01 (completeness: 93.8%)
  Volume: 4.88M MT
  3-month rolling avg (complete months): 5.12M MT
  Ratio vs 5Y baseline:                  0.857x

Corn:
  Total months available: 52
  Complete months (>=90%): 36
  Last complete month: 2024-12-01 (completeness: 94.5%)
  Volume: 5.67M MT
  3-month rolling avg (complete months): 5.86M MT
  Ratio vs 5Y baseline:                  0.839x

Rice:
  Total months available: 53
  Complete months (>=90%): 36
  Last complete month: 2024-12-01 (completeness: 95.7%)
  Volume: 1.88M MT
  3-month rolling avg (complete months): 1.49M MT
  Ratio vs 5Y baseline:                  1.183x

Soybean:
  Total months available: 65
  Complete months (>=90%): 37
  Last complete month: 2024-01-01 (completeness: 92.0%)
  Volume: 28.47M MT
  3-month rolling avg (complete months): 30.74M MT
  Ratio vs 5Y baseline:    

_**Consolidate + cap forecasts**_

In [6]:
# ══════════════════════════════════════════════════════════════════
# STEP 5: CONSOLIDATE AND SAVE PROPHET OUTPUTS
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 5: Consolidating outputs ===")

# Validation table
df_validation_all = pd.concat(
    [v for v in all_validations if v is not None],
    ignore_index=True
)
df_validation_all["ds"] = pd.to_datetime(df_validation_all["ds"])
df_validation_all = df_validation_all.sort_values(
    ["commodity","indicator","ds"]
).reset_index(drop=True)

# Forecast table
df_forecast_all = pd.concat(
    [f for f in all_forecasts if f is not None],
    ignore_index=True
)
df_forecast_all["ds"] = pd.to_datetime(df_forecast_all["ds"])
df_forecast_all = df_forecast_all.sort_values(
    ["commodity","indicator","ds"]
).reset_index(drop=True)

# Cap forecasts to physical bounds
bounds = {
    "stu_deviation": (-15.0,  10.0),
    "bdi_ratio":     (  0.5,   3.0),
    "ppi_deviation": (-15.0,  15.0),
    "ksa_deviation": (-30.0,  30.0),
}
for indicator, (lo, hi) in bounds.items():
    mask = df_forecast_all["indicator"] == indicator
    for col in ["yhat","yhat_lower","yhat_upper"]:
        df_forecast_all.loc[mask, col] = \
            df_forecast_all.loc[mask, col].clip(lo, hi)

# ── Cap FOB Price forecasts relative to each commodity's own baseline (log-space) ──
# UPDATED (Venkat) — yhat is now log(price), since Prophet trains directly on
# log-price rather than a pre-subtracted deviation. Bounds must therefore be
# computed in price-space first (50%-150% of 5Y baseline price), then converted
# to log-space before clipping — clipping raw dollar amounts against a log-scale
# yhat would be dimensionally meaningless and clip every forecast to nothing.
PRICE_BOUND_PCT = 0.50   # cap forecast price at 50%-150% of 5Y baseline price

price_mask = df_forecast_all["indicator"] == "fob_price_deviation"
for commodity in COMMODITIES:
    b_row = df_baseline[
        (df_baseline["commodity"]==commodity) &
        (df_baseline["indicator"]=="fob_price_deviation")
    ]
    if b_row.empty:
        continue
    b = float(b_row["baseline_avg"].values[0])
    lo_price, hi_price = (1 - PRICE_BOUND_PCT) * b, (1 + PRICE_BOUND_PCT) * b
    lo_log, hi_log = np.log(max(lo_price, 1e-6)), np.log(hi_price)   # NEW — convert price bounds to log-space
    c_mask = price_mask & (df_forecast_all["commodity"] == commodity)
    for col in ["yhat","yhat_lower","yhat_upper"]:
        df_forecast_all.loc[c_mask, col] = df_forecast_all.loc[c_mask, col].clip(lo_log, hi_log)   # CHANGED — clip against log bounds

# ── Round everything once, after all capping is complete ──
for col in ["yhat","yhat_lower","yhat_upper"]:
    df_forecast_all[col] = df_forecast_all[col].round(4)

# Metrics table
df_metrics = pd.DataFrame(all_metrics)
df_metrics  = df_metrics.sort_values(
    ["indicator","commodity"]
).reset_index(drop=True)

# Save
save_to_lakehouse(df_validation_all, "prophet_validation")
save_to_lakehouse(df_forecast_all,   "prophet_forecasts")
save_to_lakehouse(df_metrics,        "prophet_model_metrics")

print(f"\nValidation: {df_validation_all.shape}")
print(f"Forecasts:  {df_forecast_all.shape}")
print(f"Metrics:    {df_metrics.shape}")

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 8, Finished, Available, Finished, False)


=== Step 5: Consolidating outputs ===
✓ srm.prophet_validation: 140 rows saved
✓ srm.prophet_forecasts: 75 rows saved
✓ srm.prophet_model_metrics: 25 rows saved

Validation: (140, 10)
Forecasts:  (75, 7)
Metrics:    (25, 7)


In [7]:
# ══════════════════════════════════════════════════════════════════
# STEP 6: CURRENT VALUES — driven by SCORING_END
# Actual values for current month scoring
# ══════════════════════════════════════════════════════════════════

CURRENT_MONTH = SCORING_END
print(f"\n=== Step 6: Current values {CURRENT_MONTH.strftime('%b %Y')} ===")

CURRENT_MONTH = SCORING_END
current_rows  = []

ts_map = {
    "stu_deviation": df_stu_ts,
    "bdi_ratio":     df_bdi_ts,
    "ppi_deviation": df_ppi_ts,
    "ksa_deviation": df_ksa_ts,
    "policy_risk_weighted": df_policy_ts,
    "fob_price_deviation":  df_price_ts   # NEW
}

for indicator, df_ts in ts_map.items():
    for commodity in COMMODITIES:
        dc = df_ts[df_ts["commodity"]==commodity].copy()
        dc["ds"] = pd.to_datetime(dc["ds"])
        dc = dc.sort_values("ds")

        latest = dc[dc["ds"] <= CURRENT_MONTH].iloc[-1]
        current_rows.append({
            "ds":          CURRENT_MONTH,
            "yhat":        latest["y"],
            "yhat_lower":  latest["y"],
            "yhat_upper":  latest["y"],
            "y_raw":       latest["y_raw"],
            "baseline_avg":latest["baseline_avg"] if "baseline_avg" in latest.index else np.nan,
            "indicator":   indicator,
            "commodity":   commodity,
            "period_type": "current"
        })

# Add demand current values from rolling avg
for commodity in COMMODITIES:
    dc_demand = df_demand_current[
        (df_demand_current["commodity"]==commodity) &
        (df_demand_current["period_type"]=="current")
    ]
    if not dc_demand.empty:
        row = dc_demand.iloc[0]
        current_rows.append({
            "ds":          CURRENT_MONTH,
            "yhat":        row["yhat"],
            "yhat_lower":  row["yhat_lower"],
            "yhat_upper":  row["yhat_upper"],
            "y_raw":       row["y_raw_mt"],
            "baseline_avg":row["baseline_avg"],
            "indicator":   "demand_ratio",
            "commodity":   commodity,
            "period_type": "current"
        })

df_current = pd.DataFrame(current_rows)
df_current["ds"] = pd.to_datetime(df_current["ds"])

print(f"\nCurrent values {CURRENT_MONTH.strftime('%b %Y')}:")
col_header = "".join(f"{c:>12}" for c in COMMODITIES)
print(f"\n{'Indicator':<25}{col_header}  {'Units'}")
print("-" * (25 + 12*len(COMMODITIES) + 10))
for indicator in ["stu_deviation","bdi_ratio","demand_ratio",
                  "ppi_deviation","ksa_deviation","policy_risk_weighted", "fob_price_deviation"]:
    vals = {}
    for commodity in COMMODITIES:
        row = df_current[
            (df_current["indicator"]==indicator) &
            (df_current["commodity"]==commodity)
        ]
        vals[commodity] = row["yhat"].values[0] if not row.empty else np.nan

    units = {
        "stu_deviation":"% dev",
        "bdi_ratio":"ratio",
        "demand_ratio":"ratio",
        "ppi_deviation":"pts dev",
        "ksa_deviation":"% dev",
        "policy_risk_weighted":"0-100",
        "fob_price_deviation":"log-price",   # NEW
    }.get(indicator,"")

    row_vals = "".join(f"{vals.get(c, np.nan):>12.3f}" for c in COMMODITIES)
    print(f"{indicator:<25}{row_vals}  {units}")

save_to_lakehouse(df_current, "prophet_current_values")

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 9, Finished, Available, Finished, False)


=== Step 6: Current values Jul 2026 ===

Current values Jul 2026:

Indicator                       Wheat        Corn        Rice     Soybean      Barley  Units
-----------------------------------------------------------------------------------------------
stu_deviation                  -0.612      -3.435       0.955       0.013       1.308  % dev
bdi_ratio                       1.527       1.527       1.527       1.527       1.527  ratio
demand_ratio                    0.857       0.839       1.183       1.114       0.785  ratio
ppi_deviation                  -1.393      -0.331      -1.499       1.561       3.626  pts dev
ksa_deviation                   5.214       0.648       0.764       1.404     -25.864  % dev
policy_risk_weighted           17.508       4.949      10.844       5.829       0.000  0-100
fob_price_deviation             5.423       5.342       5.730       6.183       5.293  log-price
✓ srm.prophet_current_values: 35 rows saved


In [8]:
# ══════════════════════════════════════════════════════════════════
# STEP 7: METRICS REVIEW + FORECAST PREVIEW
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 7: Model performance review ===")

print(f"\n{'Indicator':<22} {'Commodity':<8} {'RMSE':>8} {'MAE':>8} "
      f"{'MAPE%':>8} {'Confidence':>12}")
print("-" * 70)

for _, row in df_metrics.iterrows():
    mape = row["MAPE"]
    if pd.isna(mape):
        conf = "No val data"
    elif mape < 10:
        conf = "HIGH ✓"
    elif mape < 25:
        conf = "MEDIUM"
    else:
        conf = "LOW ⚠"
    print(f"{row['indicator']:<22} {row['commodity']:<8} "
          f"{row['RMSE']:>8.3f} {row['MAE']:>8.3f} "
          f"{row['MAPE']:>7.1f}% {conf:>12}")

fm_labels = [m.strftime('%b %Y') for m in FORECAST_MONTHS]
print(f"\n=== Forecast values {fm_labels[0]}-{fm_labels[-1]} ===")
for commodity in COMMODITIES:
    dc = df_forecast_all[df_forecast_all["commodity"]==commodity]
    print(f"\n{commodity}:")
    print(f"{'Indicator':<22} {fm_labels[0]:>10} {fm_labels[1]:>10} {fm_labels[2]:>10}")
    print("-" * 55)
    for indicator in dc["indicator"].unique():
        di = dc[dc["indicator"]==indicator].sort_values("ds")
        vals = di["yhat"].values
        if len(vals) == 3:
            print(f"{indicator:<22} {vals[0]:>10.3f} {vals[1]:>10.3f} {vals[2]:>10.3f}")

print(f"\nDemand forecast (rolling avg - same for all periods):")
for commodity in COMMODITIES:
    dc = df_demand_current[
        (df_demand_current["commodity"]==commodity) &
        (df_demand_current["period_type"]=="forecast")
    ]
    if not dc.empty:
        ratio = dc["yhat"].values[0]
        print(f"  {commodity}: ratio={ratio:.3f} "
              f"({'above' if ratio > 1 else 'below'} 5Y baseline)")

val_end_label  = SCORING_END.strftime('%b %Y')
fc_start_label = FORECAST_MONTHS[0].strftime('%b')
fc_end_label   = FORECAST_MONTHS[-1].strftime('%b %Y')

print(f"\n=== NOTEBOOK 2 COMPLETE ===")
print(f"""
Tables saved:
  srm.prophet_validation       — actual vs forecast Jan-{val_end_label}
  srm.prophet_forecasts        — {fc_start_label}-{fc_end_label} Prophet output
  srm.prophet_model_metrics    — RMSE, MAE, MAPE per model
  srm.prophet_current_values   — {val_end_label} actual signals
  srm.prophet_demand_current   — demand rolling avg (no Prophet)

Next: Notebook 3 — Indicator Scoring
  New thresholds based on deviation/ratio signals
""")

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 10, Finished, Available, Finished, False)


=== Step 7: Model performance review ===

Indicator              Commodity     RMSE      MAE    MAPE%   Confidence
----------------------------------------------------------------------
bdi_ratio              Barley      0.257    0.205    13.6%       MEDIUM
bdi_ratio              Corn        0.257    0.205    13.6%       MEDIUM
bdi_ratio              Rice        0.257    0.205    13.6%       MEDIUM
bdi_ratio              Soybean     0.257    0.205    13.6%       MEDIUM
bdi_ratio              Wheat       0.257    0.205    13.6%       MEDIUM
fob_price_deviation    Barley      0.138    0.097     1.8%       HIGH ✓
fob_price_deviation    Corn        0.024    0.021     0.4%       HIGH ✓
fob_price_deviation    Rice        0.361    0.314     5.6%       HIGH ✓
fob_price_deviation    Soybean     0.111    0.096     1.6%       HIGH ✓
fob_price_deviation    Wheat       0.027    0.024     0.5%       HIGH ✓
ksa_deviation          Barley        nan      nan     nan%  No val data
ksa_deviation        

In [9]:
# ══════════════════════════════════════════════════════════════════
# NOTEBOOK 2 — ADD TO END: ENRICH VALIDATION TABLE
# Add baseline values + raw predicted values for all indicators
# So the table shows both signal (ratio/deviation) AND
# actual meaningful values (%, index, MT)
# ══════════════════════════════════════════════════════════════════

print("\n=== Enriching validation table with raw values ===")

# ── Load tables ────────────────────────────────────────────────────
df_val_raw    = load_table("prophet_validation")
df_baseline   = load_table("prophet_baseline_summary")

# ── Build baseline lookup ──────────────────────────────────────────
baseline_lookup = {}
for _, row in df_baseline.iterrows():
    baseline_lookup[(row["commodity"], row["indicator"])] = float(row["baseline_avg"])

print(f"Validation rows: {df_val_raw.shape}")
print(f"Baseline entries: {len(baseline_lookup)}")

# ── Conversion functions — signal → raw meaningful value ───────────
def signal_to_raw(value, indicator, commodity, baseline_lookup):
    """
    Convert Prophet signal (deviation/ratio) back to
    the original meaningful unit for each indicator.

    Returns: (raw_value, unit_label)
    """
    if pd.isna(value):
        return np.nan, "N/A"

    if indicator == "stu_deviation":
        # deviation + baseline = actual STU %
        baseline = baseline_lookup.get((commodity,"stu_ratio"), 26.86)
        return round((value / baseline) * 100, 3), "STU deviation %"

    elif indicator == "bdi_ratio":
        # ratio × baseline = actual BDI index value
        baseline = baseline_lookup.get((commodity,"bdi_ratio"), 1937.67)
        return round(value * baseline, 1), "BDI index"

    elif indicator == "demand_ratio":
        # ratio × baseline = actual import volume MT
        baseline = baseline_lookup.get((commodity,"demand_ratio"), 1.0)
        return round(value * baseline / 1e6, 3), "Million MT"

    elif indicator == "ppi_deviation":
        # deviation + baseline - 100 = PPI deviation from 100
        # (boss's format: values like 3, 0, -3, -6)
        baseline = baseline_lookup.get((commodity,"ppi_deviation"), 100.0)
        return round(value + baseline - 100, 3), "PPI dev from 100"

    elif indicator == "ksa_deviation":
        # deviation + baseline = actual KSA top3 share %
        baseline = baseline_lookup.get(
            (commodity,"ksa_top3_pct"),
            baseline_lookup.get((commodity,"ksa_deviation"), 85.0)
        )
        return round(min(value + baseline, 100.0), 2), "KSA top3 %"

    elif indicator == "fob_price_deviation":
        price = np.exp(value)   # CHANGED — value is now log(price), reconstruct via exp(), no baseline needed at all
        return round(price, 2), "FOB price (USD/mt)"

    else:
        return round(value, 4), "raw score"


def get_baseline_display(indicator, commodity, baseline_lookup):
    """
    Return the baseline value in its original meaningful unit.
    """
    if indicator == "stu_deviation":
        return 0.0, "STU deviation % (baseline = 0 by definition)"

    elif indicator == "bdi_ratio":
        b = baseline_lookup.get((commodity,"bdi_ratio"), 1937.67)
        return round(b, 1), "BDI index"

    elif indicator == "demand_ratio":
        b = baseline_lookup.get((commodity,"demand_ratio"), 1.0)
        return round(b / 1e6, 3), "Million MT"

    elif indicator == "ppi_deviation":
        b = baseline_lookup.get((commodity,"ppi_deviation"), 100.0)
        return round(b, 3), "PPI index (=100 baseline)"

    elif indicator == "ksa_deviation":
        b = baseline_lookup.get(
            (commodity,"ksa_top3_pct"),
            baseline_lookup.get((commodity,"ksa_deviation"), 85.0)
        )
        return round(b, 2), "KSA top3 %"

    elif indicator == "fob_price_deviation":
        b = baseline_lookup.get((commodity,"fob_price_deviation"), 250.0)
        return round(b, 2), "FOB price baseline (USD/mt)"

    else:
        return np.nan, "N/A"


# ── Apply conversions to every row ─────────────────────────────────
enriched_rows = []

for _, row in df_val_raw.iterrows():
    indicator = row["indicator"]
    commodity = row["commodity"]

    # Baseline in meaningful units
    baseline_raw, unit = get_baseline_display(indicator, commodity, baseline_lookup)

    # Actual observed value in meaningful units
    actual_raw, _    = signal_to_raw(row["y"],    indicator, commodity, baseline_lookup)

    # Prophet predicted value in meaningful units
    predicted_raw, _ = signal_to_raw(row["yhat"], indicator, commodity, baseline_lookup)

    # Lower and upper bounds in meaningful units
    lower_raw, _     = signal_to_raw(row["yhat_lower"], indicator, commodity, baseline_lookup)
    upper_raw, _     = signal_to_raw(row["yhat_upper"], indicator, commodity, baseline_lookup)

    # Residual in meaningful units
    residual_raw     = round(actual_raw - predicted_raw, 4) if not pd.isna(actual_raw) and not pd.isna(predicted_raw) else np.nan
    abs_error_raw    = round(abs(residual_raw), 4) if not pd.isna(residual_raw) else np.nan

    enriched_rows.append({
        # Original signal columns
        "ds":                  row["ds"],
        "indicator":           indicator,
        "commodity":           commodity,
        "y_signal":            round(float(row["y"]),    6),
        "yhat_signal":         round(float(row["yhat"]), 6),
        "residual_signal":     round(float(row["y"] - row["yhat"]), 6),
        "abs_error_signal":    round(abs(float(row["y"] - row["yhat"])), 6),

        # Baseline in meaningful units
        "baseline_raw":        baseline_raw,
        "unit":                unit,

        # Actual + predicted in meaningful units
        "actual_raw":          actual_raw,
        "predicted_raw":       predicted_raw,
        "predicted_lower_raw": lower_raw,
        "predicted_upper_raw": upper_raw,
        "residual_raw":        residual_raw,
        "abs_error_raw":       abs_error_raw,

        # Confidence interval width in meaningful units
        "ci_width_raw":        round(upper_raw - lower_raw, 4) if not pd.isna(upper_raw) and not pd.isna(lower_raw) else np.nan,

        # Over or under forecast flag
        "prophet_direction":   "Under-forecast" if residual_raw > 0 else "Over-forecast" if residual_raw < 0 else "Exact"
    })

df_val_enriched = pd.DataFrame(enriched_rows)
df_val_enriched["ds"] = pd.to_datetime(df_val_enriched["ds"])
df_val_enriched = df_val_enriched.sort_values(
    ["indicator","commodity","ds"]
).reset_index(drop=True)

print(f"\nEnriched validation table: {df_val_enriched.shape}")
print(f"Columns: {df_val_enriched.columns.tolist()}")

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 11, Finished, Available, Finished, False)


=== Enriching validation table with raw values ===
Validation rows: (140, 10)
Baseline entries: 30

Enriched validation table: (140, 17)
Columns: ['ds', 'indicator', 'commodity', 'y_signal', 'yhat_signal', 'residual_signal', 'abs_error_signal', 'baseline_raw', 'unit', 'actual_raw', 'predicted_raw', 'predicted_lower_raw', 'predicted_upper_raw', 'residual_raw', 'abs_error_raw', 'ci_width_raw', 'prophet_direction']


In [10]:
# ── Print enriched validation per indicator ────────────────────────
print("\n=== Enriched validation — actual vs predicted in raw units ===")

for indicator in ["stu_deviation","bdi_ratio","ppi_deviation","ksa_deviation","fob_price_deviation"]:
    print(f"\n{'─'*105}")
    print(f"{indicator.upper()}")
    print(f"{'─'*105}")

    for commodity in COMMODITIES:
        dc = df_val_enriched[
            (df_val_enriched["indicator"]==indicator) &
            (df_val_enriched["commodity"]==commodity)
        ].sort_values("ds")

        if dc.empty:
            continue

        unit        = dc["unit"].values[0]
        baseline    = dc["baseline_raw"].values[0]

        print(f"\n  {commodity} — Unit: {unit} | 5Y Baseline: {baseline}")
        print(f"  {'Month':<12} {'Actual':>12} {'Predicted':>12} "
              f"{'Lower':>12} {'Upper':>12} {'Residual':>12} {'Direction'}")
        print(f"  {'-'*85}")

        for _, row in dc.iterrows():
            print(f"  {str(row['ds'].date()):<12} "
                  f"{row['actual_raw']:>12.2f} "
                  f"{row['predicted_raw']:>12.2f} "
                  f"{row['predicted_lower_raw']:>12.2f} "
                  f"{row['predicted_upper_raw']:>12.2f} "
                  f"{row['residual_raw']:>12.2f} "
                  f"{row['prophet_direction']}")

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 12, Finished, Available, Finished, False)


=== Enriched validation — actual vs predicted in raw units ===

─────────────────────────────────────────────────────────────────────────────────────────────────────────
STU_DEVIATION
─────────────────────────────────────────────────────────────────────────────────────────────────────────

  Wheat — Unit: STU deviation % (baseline = 0 by definition) | 5Y Baseline: 0.0
  Month              Actual    Predicted        Lower        Upper     Residual Direction
  -------------------------------------------------------------------------------------
  2026-01-01          -0.75        -2.68        -4.75        -0.42         1.93 Under-forecast
  2026-02-01          -1.24        -2.32        -4.55        -0.21         1.09 Under-forecast
  2026-03-01          -1.53        -1.98        -4.03         0.26         0.45 Under-forecast
  2026-04-01           1.15        -1.58        -3.59         0.44         2.73 Under-forecast
  2026-05-01          -1.05        -1.18        -3.28         1.07    

In [12]:
# ══════════════════════════════════════════════════════════════════
# DIAGNOSTIC: FOB Price changepoint_prior_scale grid search (log-price signal)
# One-time tuning run — not part of the final Step 3e block.
# Tests multiple values per commodity, reports RMSE in actual USD/mt
# (not log-scale, which is misleadingly small regardless of fit quality).
# ══════════════════════════════════════════════════════════════════

import numpy as np

CANDIDATE_SCALES = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]

print("\n=== FOB Price changepoint_prior_scale grid search ===")
print(f"{'Commodity':<10}{'Scale':>8}{'RMSE ($)':>12}{'MAE ($)':>12}{'Jul Residual':>15}")
print("-" * 60)

tuning_results = []

for commodity in COMMODITIES:
    for scale in CANDIDATE_SCALES:
        val, fore, metrics = train_prophet(
            df_price_ts, commodity, "fob_price_deviation",
            yearly_seasonality=True,
            changepoint_prior_scale=scale,
            seasonality_prior_scale=10.0
        )
        if val is None or val.empty:
            continue

        val_sorted = val.sort_values("ds")
        actual_price    = np.exp(val_sorted["y"].values)
        predicted_price = np.exp(val_sorted["yhat"].values)
        residuals = actual_price - predicted_price

        rmse = np.sqrt(np.mean(residuals**2))
        mae  = np.mean(np.abs(residuals))
        jul_residual = residuals[-1]   # last validation month — the one closest to the forecast edge

        tuning_results.append({
            "commodity": commodity, "scale": scale,
            "rmse": rmse, "mae": mae, "jul_residual": jul_residual
        })
        print(f"{commodity:<10}{scale:>8.2f}{rmse:>12.2f}{mae:>12.2f}{jul_residual:>15.2f}")

    print("-" * 60)

df_tuning = pd.DataFrame(tuning_results)
print("\n=== Best scale per commodity (lowest RMSE) ===")
best = df_tuning.loc[df_tuning.groupby("commodity")["rmse"].idxmin()]
print(best[["commodity","scale","rmse","mae","jul_residual"]].to_string(index=False))

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 15, Finished, Available, Finished, False)

01:52:30 - cmdstanpy - INFO - Chain [1] start processing
01:52:30 - cmdstanpy - INFO - Chain [1] done processing
01:52:31 - cmdstanpy - INFO - Chain [1] start processing
01:52:31 - cmdstanpy - INFO - Chain [1] done processing
01:52:31 - cmdstanpy - INFO - Chain [1] start processing
01:52:31 - cmdstanpy - INFO - Chain [1] start processing
01:52:31 - cmdstanpy - INFO - Chain [1] done processing
01:52:33 - cmdstanpy - INFO - Chain [1] done processing
01:52:35 - cmdstanpy - INFO - Chain [1] done processing
01:52:35 - cmdstanpy - INFO - Chain [1] start processing
01:52:35 - cmdstanpy - INFO - Chain [1] done processing


    RMSE=0.313  MAE=0.262  MAPE=4.7%
    Forecast: [5.134 5.087 5.04 ]
Rice          0.01       73.07       60.60         130.29

  fob_price_deviation | Rice
    Train: 84 rows (2019-01-01 → 2025-12-01)
    Val:   7 rows
    RMSE=0.145  MAE=0.104  MAPE=1.9%
    Forecast: [5.564 5.576 5.588]
Barley        0.30       33.10       24.04         -58.82

  fob_price_deviation | Barley
    Train: 100 rows (2017-09-01 → 2025-12-01)
    Val:   7 rows
    RMSE=0.144  MAE=0.104  MAPE=1.9%
    Forecast: [5.563 5.575 5.586]
Barley        0.50       32.98       23.95         -58.60
------------------------------------------------------------

=== Best scale per commodity (lowest RMSE) ===
commodity  scale      rmse       mae  jul_residual
   Barley   0.01 18.297769 16.460540    -25.943546
     Corn   0.01  4.825376  4.310163      4.163696
     Rice   0.01 73.066760 60.600547    130.294458
  Soybean   0.50 29.469991 24.643172     56.997096
    Wheat   0.05  6.437105  5.714918     -9.872240


In [11]:
# ── Save enriched validation table ────────────────────────────────
save_to_lakehouse(df_val_enriched, "prophet_validation_enriched")

print(f"""
=== Enriched validation table saved ===

Table: srm.prophet_validation_enriched

Columns added vs original prophet_validation:
  baseline_raw         — 5Y baseline in original units
  unit                 — unit label (STU%, BDI index, etc)
  actual_raw           — actual observed value in original units
  predicted_raw        — Prophet's prediction in original units
  predicted_lower_raw  — lower confidence bound in original units
  predicted_upper_raw  — upper confidence bound in original units
  residual_raw         — actual - predicted in original units
  abs_error_raw        — |residual| in original units
  ci_width_raw         — confidence interval width
  prophet_direction    — Under-forecast / Over-forecast / Exact
  y_signal             — original ratio/deviation signal
  yhat_signal          — original predicted signal

Example for BDI (Jan 2026 Corn):
  baseline_raw    = 1,937.67  (5Y avg BDI index)
  actual_raw      = 1,782     (actual BDI in Jan 2026)
  predicted_raw   = 1,949     (Prophet predicted BDI)
  residual_raw    = -167      (actual was 167 pts below prediction)
  unit            = BDI index
""")

StatementMeta(, d8f76def-9120-412f-b7d3-c91657b88137, 13, Finished, Available, Finished, False)

✓ srm.prophet_validation_enriched: 140 rows saved

=== Enriched validation table saved ===

Table: srm.prophet_validation_enriched

Columns added vs original prophet_validation:
  baseline_raw         — 5Y baseline in original units
  unit                 — unit label (STU%, BDI index, etc)
  actual_raw           — actual observed value in original units
  predicted_raw        — Prophet's prediction in original units
  predicted_lower_raw  — lower confidence bound in original units
  predicted_upper_raw  — upper confidence bound in original units
  residual_raw         — actual - predicted in original units
  abs_error_raw        — |residual| in original units
  ci_width_raw         — confidence interval width
  prophet_direction    — Under-forecast / Over-forecast / Exact
  y_signal             — original ratio/deviation signal
  yhat_signal          — original predicted signal

Example for BDI (Jan 2026 Corn):
  baseline_raw    = 1,937.67  (5Y avg BDI index)
  actual_raw      = 1,782